# Environment

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import re
import os
import time
import logging
import networkx as nx
import kbase.preprocessing as dp  

from pathlib import Path
from typing import List, Tuple, Dict
from __future__ import annotations
from typing import List, Tuple, Dict
from tqdm.auto import tqdm

import matplotlib.pyplot as plt


# Data Loading

In [ ]:
df_edatoscaso = pq.read_table(os.path.join(dp.source_tables_path, 
                                           "EDATOSCASO_source/CCR_HIST_EDATOSCASO_LLAMADAS_ANONYMISED_01_12_2025.parquet")).to_pandas()


# Main vectors

In [ ]:
key_col = 'DEMANDAPK'
id_columns = ['IDTARJETASANITARIA', 'DNI', 'NOMBRE_APELLIDOS']

# Run pipeline in subset 

In [ ]:
# Flag to test the pipeline in a subset of 100K
## must be either "all", to run in all dataframe, or "test", to run it in the first 100K rows
df_edatoscaso = dp.df_pipeline_test(df_edatoscaso, "test").sort_values(by="DEMANDAPK")

In [ ]:
# df_edatoscaso["NOMBRE_APELLIDOS"] = df_edatoscaso["NOMBRE_APELLIDOS"].apply(
#     lambda x: None if isinstance(x, str) and "DESC" in x.upper() else x
# )

In [ ]:
df_edatoscaso.head(100)

In [ ]:
df_edatoscaso.info()

In [ ]:

def combine_dni_columns(df, col_dni_affected, col_dni, new_col):
    """
    Combine two DNI columns into a single one following priority rules:
    
    Rules:
    1. If both values are None/NaN -> result is None
    2. If one value is None/NaN -> use the valid one
    3. If both are valid and equal -> use that value
    4. If both are valid but different -> 
         - take value from col_dni_affected
         - count such conflicts and print number of affected rows
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    col_dni_affected : str
        Name of the DNI column from EDATOSCASO table.
    col_dni : str
        Name of the DNI column from LLAMADAS table
    new_col : str
        Name of the resulting combined column.
    
    Returns
    -------
    df : pd.DataFrame
        Dataframe including the new combined column.
    """

    # Identify conflicting rows: both non-null AND different
    conflict_mask = (
        df[col_dni_affected].notna() &
        df[col_dni].notna() &
        (df[col_dni_affected] != df[col_dni])
    )
    
    # Print total conflicts
    num_conflicts = conflict_mask.sum()
    if num_conflicts > 0:
        print(f"[INFO] Number of rows where DNI columns differ: {num_conflicts}")
    else:
        print("[INFO] No conflicts detected between DNI columns.")
    
    # Apply combination logic
    def combine_row(a, b):
        # Case 1: both None
        if pd.isna(a) and pd.isna(b):
            return None
        
        # Case 2: one valid, one None
        if pd.isna(a) and not pd.isna(b):
            return b
        if not pd.isna(a) and pd.isna(b):
            return a
        
        # Case 3: both valid and equal
        if a == b:
            return a
        
        # Case 4: both valid but different -> take the affected DNI
        return a  # col_dni_affected
        
    df[new_col] = df.apply(lambda row: combine_row(row[col_dni_affected], row[col_dni]), axis=1)
    
    return df

# Call the function
df_edatoscaso = combine_dni_columns(df_edatoscaso, col_dni_affected="DNIAFECTADO", col_dni="DNI", new_col="DNI_final")

# Remove old columns
df_edatoscaso = df_edatoscaso.drop(columns=["DNIAFECTADO", "DNI"])

# Rename DNI_final → DNI
df_edatoscaso = df_edatoscaso.rename(columns={"DNI_final": "DNI"})



In [ ]:
df_edatoscaso = dp.reorder_dataframe(df_edatoscaso)

In [ ]:
df_edatoscaso.head(5)

In [ ]:
dp.summarize_numbers(df_edatoscaso, key_col, id_columns)
print(f"\nShape of df_edatoscaso: {df_edatoscaso.shape}")

num_rows = dp.count_rows_all_none(df_edatoscaso, id_columns)

# Removing rows None for all ID (except in single DEMANDAPK)

In [ ]:
def drop_duplicate_none_rows(df, key_column, null_columns):
    """
    Removes rows where the key column is duplicated AND the specified columns 
    contain None/NaN in ALL of them.

    Args:
        df (pd.DataFrame): The input DataFrame.
        key_column (str): The name of the column used to check for duplicates (e.g., 'DEMANDAPK').
        null_columns (list): List of columns to check for null values (e.g., ['IDTARJETASANITARIA', 'DNI', 'NOMBRE_APELLIDOS']).

    Returns:
        pd.DataFrame: The DataFrame with the duplicated and null rows removed.
    """
    # 1. Identify ALL rows that have a duplicate in the key column
    # keep=False ensures both the first and subsequent instances of a duplicate are marked True
    duplicate_mask = df[key_column].duplicated(keep=False)

    # 2. Identify rows where ALL 3 target columns are None/NaN
    # .isnull() works for detecting both None (in object type) and np.nan (in float type)
    # .all(axis=1) ensures it's True only if ALL three are True across the row
    none_mask = df[null_columns].isnull().all(axis=1)

    # 3. Combine the masks: True if it's a duplicate AND if ALL 3 columns are None
    rows_to_drop = duplicate_mask & none_mask

    # 4. Drop the identified rows
    # We use .drop() which returns a new DataFrame (it does not modify the original in-place)
    df_clean = df.drop(df[rows_to_drop].index)

    # Optional: Print the summary within the function
    print(f"Original shape: {df.shape}")
    print(f"Rows dropped (duplicated and null): {rows_to_drop.sum()}")
    print(f"New shape: {df_clean.shape}")

    return df_clean

# Call the function.
df_edatoscaso_filterNaN = drop_duplicate_none_rows(
    df=df_edatoscaso, 
    key_column=key_col, 
    null_columns=id_columns
)

In [ ]:
dp.summarize_numbers(df_edatoscaso_filterNaN, key_col, id_columns)
print(f"\nShape of df_edatoscaso_filterNaN: {df_edatoscaso_filterNaN.shape}")

num_rows = dp.count_rows_all_none(df_edatoscaso_filterNaN, id_columns)

In [ ]:
df_edatoscaso_filterNaN.head(10)

# Grouping of Duplicates

In [ ]:
"""
Duplicate resolution function for DEMANDAPK records by person identity.

This module provides functionality to identify and merge duplicate records
for the same person (identified by NUHSA, DNI, or Name) while preserving
distinct values in specified columns.

"""
# ============================================================================
# CONFIGURATION
# ============================================================================

# Configure logging
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# Primary identifier column for grouping duplicates
ID_COL = key_col

# Person identity columns (in priority order for base row selection)
PERSON_ID_COLS = id_columns

# Columns whose values should be combined when both rows have the same identity
COMBINE_COLS: List[str] = [
    "IDDEMANDA",
    "IDRECURSO", 
    "IDASISTENCIA", 
    "IDTIPORECURSO",
    "IDTIPORECURSOAGR",
    "IDEMPRESA",
    "IDCODIGORESOLUCION",
    "IDCODRESOLUCIONRECURSO",
    "JUICIOCLINICO1",
    "JUICIOCLINICO2",
    "JUICIOCLINICO3",
    "PRIORIDAD",
]

# Separator used when combining distinct values
COMBINE_SEP: str = " | "

# ============================================================================
# HELPER FUNCTIONS: VALUE CHECKING AND COMBINING
# ============================================================================

def _is_missing(value) -> bool:
    """
    Check if a value is considered missing.

    Handles multiple representations of missing data:
    - pandas NA/NaN values
    - None values
    - Empty or whitespace-only strings

    Parameters
    ----------
    value : any
        Value to check

    Returns
    -------
    bool
        True if value is missing, False otherwise

    Examples
    --------
    >>> _is_missing(pd.NA)
    True
    >>> _is_missing("   ")
    True
    >>> _is_missing("valid")
    False
    """
    if pd.isna(value):
        return True
    if isinstance(value, str):
        return value.strip() == ""
    return False


def _normalize_id_value(value) -> Optional[str]:
    """
    Normalize an identity value for comparison across records.

    Normalization includes:
    - Converting to uppercase for case-insensitive matching
    - Stripping whitespace
    - Returning None for missing values

    Parameters
    ----------
    value : any
        Identity value to normalize

    Returns
    -------
    Optional[str]
        Normalized string or None if missing

    Examples
    --------
    >>> _normalize_id_value("  john DOE  ")
    'JOHN DOE'
    >>> _normalize_id_value(pd.NA)
    None
    """
    if _is_missing(value):
        return None
    normalized = str(value).strip().upper()
    return normalized if normalized else None


def _combine_two_values(value1, value2, sep: str = COMBINE_SEP) -> object:
    """
    Combine two cell values using a separator.

    Combination rules:
    - If both values are missing: return pd.NA
    - If one value is missing: return the other value
    - If both values are identical (after stripping): return the first value
    - Otherwise: concatenate using separator

    Parameters
    ----------
    value1 : any
        First value to combine
    value2 : any
        Second value to combine
    sep : str, default " | "
        Separator string

    Returns
    -------
    object
        Combined value

    Examples
    --------
    >>> _combine_two_values("A", "B")
    'A | B'
    >>> _combine_two_values("A", pd.NA)
    'A'
    >>> _combine_two_values("A", "  A  ")
    'A'
    """
    is_missing_1 = _is_missing(value1)
    is_missing_2 = _is_missing(value2)

    # Both missing: return NA
    if is_missing_1 and is_missing_2:
        return pd.NA

    # One missing: return the other
    if is_missing_1:
        return value2
    if is_missing_2:
        return value1

    # Both present: check for equality and combine
    str_val1 = str(value1).strip()
    str_val2 = str(value2).strip()

    if str_val1 == str_val2:
        return value1

    # Normalize strings to avoid duplicate content
    return f"{str_val1}{sep}{str_val2}"


def _combine_sequence(series: pd.Series, sep: str = COMBINE_SEP) -> object:
    """
    Combine a sequence of values left-to-right.

    Efficiently combines multiple values while filtering out missing entries.
    Preserves the original row order from the input series.

    Parameters
    ----------
    series : pd.Series
        Series of values to combine
    sep : str, default " | "
        Separator string

    Returns
    -------
    object
        Combined value or pd.NA if no valid values exist

    Examples
    --------
    >>> s = pd.Series(["A", "B", "A"])
    >>> _combine_sequence(s)
    'A | B | A'
    """
    if series.empty:
        return pd.NA

    # Filter valid values and combine
    valid_values = [
        str(v).strip()
        for v in series
        if not _is_missing(v)
    ]

    if not valid_values:
        return pd.NA

    return sep.join(valid_values)


def _canonicalize_combined(value, sep: str = COMBINE_SEP) -> object:
    """
    Post-process a combined string to remove duplicates while preserving order.

    Steps:
    1. Split combined string by separator
    2. Remove empty/whitespace tokens
    3. Remove duplicate tokens (case-sensitive, preserving first occurrence)
    4. Return single value if only one token remains, else join with separator

    Parameters
    ----------
    value : any
        Combined value to canonicalize
    sep : str, default " | "
        Separator string

    Returns
    -------
    object
        Canonicalized value or pd.NA if empty

    Examples
    --------
    >>> _canonicalize_combined("A | B | A | B")
    'A | B'
    >>> _canonicalize_combined("A | A | A")
    'A'
    """
    if _is_missing(value):
        return pd.NA

    # Split and clean tokens
    parts = [p.strip() for p in str(value).split(sep)]
    parts = [p for p in parts if p]

    if not parts:
        return pd.NA

    # Remove duplicates while preserving order
    seen: Set[str] = set()
    unique_parts: List[str] = []
    for part in parts:
        if part not in seen:
            seen.add(part)
            unique_parts.append(part)

    # Return single value or joined string
    if len(unique_parts) == 1:
        return unique_parts[0]
    return sep.join(unique_parts)


# ============================================================================
# GRAPH DATA STRUCTURE: DISJOINT-SET UNION (UNION-FIND)
# ============================================================================

class DisjointSetUnion:
    """
    Disjoint-Set Union (Union-Find) data structure.

    Used to efficiently track connected components of rows that share
    person identity. Implements path compression and union by rank
    for near-constant time operations.

    Attributes
    ----------
    parent : List[int]
        Parent pointer for each element
    rank : List[int]
        Rank (height) of each component
    n : int
        Total number of elements

    Complexity
    ----------
    - find(): O(α(n)) amortized, where α is inverse Ackermann
    - union(): O(α(n)) amortized
    - get_components(): O(n)

    Examples
    --------
    >>> dsu = DisjointSetUnion(5)
    >>> dsu.union(0, 1)
    True
    >>> dsu.union(2, 3)
    True
    >>> dsu.find(0) == dsu.find(1)
    True
    """

    def __init__(self, n: int):
        """
        Initialize DSU with n elements.

        Parameters
        ----------
        n : int
            Number of elements (must be >= 0)

        Raises
        ------
        ValueError
            If n < 0
        """
        if n < 0:
            raise ValueError("Number of elements must be non-negative")
        self.parent = list(range(n))
        self.rank = [0] * n
        self.n = n

    def find(self, element: int) -> int:
        """
        Find the representative of an element's set (with path compression).

        Parameters
        ----------
        element : int
            Element to find

        Returns
        -------
        int
            Representative of element's set

        Raises
        ------
        IndexError
            If element is out of range [0, n)
        """
        if element < 0 or element >= self.n:
            raise IndexError(f"Index {element} out of range [0, {self.n})")

        if self.parent[element] != element:
            # Path compression: make element point directly to root
            self.parent[element] = self.find(self.parent[element])
        return self.parent[element]

    def union(self, element1: int, element2: int) -> bool:
        """
        Union two elements' sets (with union by rank).

        Parameters
        ----------
        element1 : int
            First element
        element2 : int
            Second element

        Returns
        -------
        bool
            True if union was performed, False if already in same set

        Raises
        ------
        IndexError
            If elements are out of range
        """
        root1 = self.find(element1)
        root2 = self.find(element2)

        if root1 == root2:
            return False

        # Union by rank: attach smaller tree under larger
        if self.rank[root1] < self.rank[root2]:
            root1, root2 = root2, root1

        self.parent[root2] = root1
        if self.rank[root1] == self.rank[root2]:
            self.rank[root1] += 1

        return True

    def get_components(self) -> List[List[int]]:
        """
        Get all connected components.

        Returns
        -------
        List[List[int]]
            List of components, each component is a list of element indices.
            Components are sorted by their minimum element.
        """
        components: Dict[int, List[int]] = {}
        for i in range(self.n):
            root = self.find(i)
            if root not in components:
                components[root] = []
            components[root].append(i)

        # Sort components by minimum element for consistency
        return sorted(components.values(), key=lambda comp: min(comp))


# ============================================================================
# IDENTITY GROUPING LOGIC
# ============================================================================

def _group_by_identity(
    df_group: pd.DataFrame,
    id_columns: List[str]
) -> List[List[int]]:
    """
    Partition rows into connected components based on shared person identity.

    Two rows are connected if they share at least one normalized value
    across any of the identity columns (NUHSA, DNI, Name).

    Algorithm:
    1. Initialize DSU with one element per row
    2. For each identity column, connect rows with identical normalized values
    3. Extract connected components from DSU

    Parameters
    ----------
    df_group : pd.DataFrame
        DataFrame group (all rows with same DEMANDAPK ID)
    id_columns : List[str]
        List of identity column names to consider

    Returns
    -------
    List[List[int]]
        List of components, each containing row indices with same identity

    Examples
    --------
    >>> df = pd.DataFrame({
    ...     'DEMANDAPK': [1, 1, 1],
    ...     'DNI': ['12345', '12345', '67890'],
    ...     'NAME': ['Alice', pd.NA, pd.NA]
    ... })
    >>> components = _group_by_identity(df, ['DNI', 'NAME'])
    >>> len(components)
    2  # Two distinct identities
    """
    df_indexed = df_group.reset_index(drop=True)
    n_rows = len(df_indexed)

    if n_rows == 0:
        return []

    dsu = DisjointSetUnion(n_rows)

    # Process each identity column
    for col in id_columns:
        if col not in df_indexed.columns:
            logger.debug(f"Identity column '{col}' not found in data")
            continue

        # Track normalized values to find duplicates
        seen_values: Dict[Optional[str], int] = {}

        for row_idx, value in enumerate(df_indexed[col]):
            normalized = _normalize_id_value(value)

            if normalized is not None:
                if normalized in seen_values:
                    # Connect rows with same normalized value
                    dsu.union(row_idx, seen_values[normalized])
                else:
                    seen_values[normalized] = row_idx

    return dsu.get_components()


# ============================================================================
# BASE ROW SELECTION
# ============================================================================

def _select_base_row(
    block: pd.DataFrame,
    id_columns: List[str]
) -> int:
    """
    Select the base row for a component using identity column priority.

    Selection order:
    1. First row with non-missing value in id_columns[0]
    2. If none found, first row with non-missing value in id_columns[1]
    3. Continue with remaining columns
    4. If no valid identity found, use first row

    Parameters
    ----------
    block : pd.DataFrame
        Subset of rows forming a single component
    id_columns : List[str]
        List of identity columns in priority order

    Returns
    -------
    int
        Index of selected base row

    Examples
    --------
    >>> df = pd.DataFrame({
    ...     'NUHSA': [pd.NA, '123', '456'],
    ...     'DNI': ['789', pd.NA, '789']
    ... })
    >>> _select_base_row(df, ['NUHSA', 'DNI'])
    1  # Row with NUHSA value has priority
    """
    for col in id_columns:
        if col not in block.columns:
            continue

        # Find rows with non-missing values in this column
        valid_mask = block[col].apply(lambda x: not _is_missing(x))

        if valid_mask.any():
            # Return index of first valid row
            return block.index[valid_mask.argmax()]

    # Fallback: return first row
    return block.index[0]


# ============================================================================
# MAIN DEDUPLICATION FUNCTION
# ============================================================================

def resolve_all_duplicates(
    df: pd.DataFrame,
    id_col: str = ID_COL,
    combine_cols: Optional[List[str]] = None,
    combine_sep: str = COMBINE_SEP,
    person_id_cols: Optional[List[str]] = None,
    progress: bool = True,
    tqdm_kwargs: Optional[Dict] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Resolve duplicate records by person identity across multiple ID columns.

    For each group of records with the same DEMANDAPK ID:
    1. Partition rows into components based on shared person identity
    2. Emit one row per component
    3. Combine values in specified columns across component rows
    4. Preserve singletons (DEMANDAPK with only 1 row) unchanged

    Parameters
    ----------
    df : pd.DataFrame
        Input dataset
    id_col : str, default "DEMANDAPK"
        Column containing primary identifier (case ID)
    combine_cols : Optional[List[str]], default None
        Columns whose values should be combined. Uses COMBINE_COLS if None.
    combine_sep : str, default " | "
        Separator for combining values
    person_id_cols : Optional[List[str]], default None
        Columns defining person identity. Uses PERSON_ID_COLS if None.
    progress : bool, default True
        Show progress bar
    tqdm_kwargs : Optional[Dict], default None
        Additional arguments for tqdm progress bar

    Returns
    -------
    final_df : pd.DataFrame
        Complete dataset with duplicates resolved
    summary : pd.DataFrame
        Summary statistics per DEMANDAPK group with columns:
        - DEMANDAPK: group identifier
        - case: resolution case ("components_person_identity")
        - input_rows: number of input rows
        - output_rows: number of output rows
        - n_components: number of identity components found

    Raises
    ------
    ValueError
        If required columns are missing
    pd.errors.EmptyDataError
        If input DataFrame is empty

    Examples
    --------
    >>> df = pd.read_csv('data.csv')
    >>> resolved_df, summary = resolve_all_duplicates(df)
    >>> print(summary[['DEMANDAPK', 'input_rows', 'output_rows']])
    """
    
    start_time = time.perf_counter()
    print("######## Starting Deduplication ########")
    # ========================================================================
    # INPUT VALIDATION
    # ========================================================================

    if df.empty:
        logger.warning("Input DataFrame is empty")
        return df.copy(), pd.DataFrame()

    if id_col not in df.columns:
        raise ValueError(
            f"Primary ID column '{id_col}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    # Set defaults
    if combine_cols is None:
        combine_cols = COMBINE_COLS
    if person_id_cols is None:
        person_id_cols = PERSON_ID_COLS

    # Validate identity columns exist
    missing_id_cols = [c for c in person_id_cols if c not in df.columns]
    if missing_id_cols:
        logger.warning(
            f"Identity columns not found: {missing_id_cols}. "
            f"Continuing with available columns."
        )
        person_id_cols = [c for c in person_id_cols if c in df.columns]

    # Validate combine columns exist
    missing_combine = [c for c in combine_cols if c not in df.columns]
    if missing_combine:
        logger.warning(
            f"Combine columns not found: {missing_combine}. "
            f"Continuing with available columns."
        )
        combine_cols = [c for c in combine_cols if c in df.columns]

    if not combine_cols:
        logger.warning("No valid combine columns found")

    # ========================================================================
    # IDENTIFY DUPLICATES AND SINGLETONS
    # ========================================================================

    value_counts = df[id_col].value_counts(dropna=False)
    duplicate_ids = value_counts.index[value_counts > 1]

    df_singletons = df[~df[id_col].isin(duplicate_ids)].copy()
    df_duplicates = df[df[id_col].isin(duplicate_ids)].copy()

    logger.info(
        f"Found {len(duplicate_ids)} DEMANDAPK groups with duplicates "
        f"({len(df_duplicates)} rows)"
    )

    # ========================================================================
    # PROCESS DUPLICATES
    # ========================================================================

    out_records: List[pd.Series] = []
    summary_rows: List[Dict] = []

    grouped = df_duplicates.groupby(id_col, sort=False)

    # Configure progress bar
    pbar_config = {
        "total": grouped.ngroups,
        "desc": "Resolving DEMANDAPK duplicates",
        "leave": True,
        "unit": "group",
        "disable": not progress,
        "dynamic_ncols": True,
    }
    if tqdm_kwargs:
        pbar_config.update(tqdm_kwargs)

    with tqdm(**pbar_config) as pbar:
        for group_id, group_df in grouped:
            try:
                # Preserve original row order
                group_sorted = group_df.sort_index()

                # Find identity components within this group
                components = _group_by_identity(group_sorted, person_id_cols)
                n_components = len(components)

                # Process each identity component
                output_count = 0
                group_indexed = group_sorted.reset_index(drop=True)

                for component in components:
                    component_block = group_indexed.iloc[component]

                    # Select base row using identity priority
                    base_row_idx = _select_base_row(component_block, person_id_cols)
                    base_row = group_indexed.loc[base_row_idx].copy()

                    # Combine specified columns across component
                    for col in combine_cols:
                        if col in component_block.columns:
                            combined_value = _combine_sequence(
                                component_block[col],
                                sep=combine_sep
                            )
                            base_row[col] = _canonicalize_combined(
                                combined_value,
                                sep=combine_sep
                            )

                    # Fill NON-combine columns from the first non-missing value within the component
                    skip_cols = set([id_col] + person_id_cols + combine_cols)
                    for col in component_block.columns:
                        if col in skip_cols:
                            continue
                        if _is_missing(base_row.get(col, pd.NA)):
                            for v in component_block[col].tolist():
                                if not _is_missing(v):
                                    base_row[col] = v
                                    break

                    out_records.append(base_row)
                    output_count += 1

                # Record summary
                summary_rows.append({
                    id_col: group_id,
                    "case": "components_person_identity",
                    "input_rows": len(group_df),
                    "output_rows": output_count,
                    "n_components": n_components,
                })

            except Exception as e:
                logger.error(
                    f"Error processing DEMANDAPK {group_id}: {str(e)}"
                )
                raise

            finally:
                pbar.update(1)

    # ========================================================================
    # COMBINE AND RETURN RESULTS
    # ========================================================================

    if out_records:
        resolved_duplicates = pd.DataFrame(out_records).reset_index(drop=True)
        final_df = pd.concat(
            [df_singletons, resolved_duplicates],
            ignore_index=True,
            sort=False
        )
    else:
        final_df = df_singletons.copy()

    # Preserve original column order
    column_order = [c for c in df.columns if c in final_df.columns]
    final_df = final_df[column_order]

    summary_df = pd.DataFrame(summary_rows) if summary_rows else pd.DataFrame()

    logger.info(
        f"Resolution complete: {len(df)} rows -> {len(final_df)} rows"
    )
    
    print(f"\nDeduplication is finished and took: {round((time.perf_counter() - start_time)/60, 2)} minutes")

    return final_df.sort_values(by='DEMANDAPK'), summary_df


# ============================================================================
# UTILITY FUNCTIONS FOR ANALYSIS
# ============================================================================

def get_resolution_statistics(summary: pd.DataFrame) -> Dict:
    """
    Calculate aggregate statistics from resolution summary.

    Parameters
    ----------
    summary : pd.DataFrame
        Summary output from resolve_all_duplicates()

    Returns
    -------
    Dict
        Statistics including total rows, reduction percentage, etc.

    Examples
    --------
    >>> stats = get_resolution_statistics(summary)
    >>> print(f"Reduction: {stats['reduction_pct']:.1f}%")
    """
    if summary.empty:
        return {
            "total_groups": 0,
            "total_input_rows": 0,
            "total_output_rows": 0,
            "reduction_pct": 0.0,
        }

    total_input = summary["input_rows"].sum()
    total_output = summary["output_rows"].sum()
    reduction = total_input - total_output

    return {
        "total_groups": len(summary),
        "total_input_rows": total_input,
        "total_output_rows": total_output,
        "rows_removed": reduction,
        "reduction_pct": (reduction / total_input * 100) if total_input > 0 else 0.0,
        "avg_components_per_group": summary["n_components"].mean(),
    }


# ============================================================================
# EXAMPLE USAGE (COMMENTED OUT)
# ============================================================================

# """
# if __name__ == "__main__":
#     # Load data
#     df = pd.read_csv("demandapk_data.csv")
    
#     # Resolve duplicates
#     resolved_df, summary = resolve_all_duplicates(df)
    
#     # Analyze results
#     stats = get_resolution_statistics(summary)
#     print(f"Resolution statistics:")
#     print(f"  Total groups: {stats['total_groups']}")
#     print(f"  Input rows: {stats['total_input_rows']}")
#     print(f"  Output rows: {stats['total_output_rows']}")
#     print(f"  Reduction: {stats['reduction_pct']:.1f}%")
    
#     # Save results
#     resolved_df.to_csv("resolved_data.csv", index=False)
#     summary.to_csv("resolution_summary.csv", index=False)
# """

In [ ]:

df_edatoscaso_grouped, summary = resolve_all_duplicates(df_edatoscaso_filterNaN)
print("\nShape of final dataframe: \n", df_edatoscaso_grouped.shape)

# Analyze results
stats = get_resolution_statistics(summary)
print(f"\nResolution statistics:")
print(f"  Total DEMANDAPK groups: {stats['total_groups']}")
print(f"  Input rows: {stats['total_input_rows']}")
print(f"  Output rows: {stats['total_output_rows']}")
print(f"  Reduction: {stats['reduction_pct']:.1f}%")
                                                        
# Optional: save outputs
# final_df.to_csv("resolved_dataset.csv", index=False)
# summary.to_csv("resolve_summary.csv", index=False)

In [ ]:
dp.summarize_numbers(df_edatoscaso_grouped, key_col, id_columns)

In [ ]:
df_edatoscaso_grouped.sort_values(by = 'DEMANDAPK').head(10)

# Creation of PATIENTID variable

In [ ]:

def create_hierarchical_patient_id(df, key_column, id_columns, birth_date_col='FNACIMIENTO'):
    """
    Creates a unique, random numeric PATIENTID for each patient group, 
    following strict hierarchical rules: P1 (IDTS) > P2 (DNI). 
    Records not linked by P1 or P2 (i.e., those with only NOMBRE_APELLIDOS 
    or having all IDs as None) will each receive a unique PATIENTID.

    Args:
        df (pd.DataFrame): The input DataFrame.
        key_column (str): The DEMANDAPK column (used as the base ID for unlinked records).
        id_columns (list): Identifier columns [IDTARJETASANITARIA, DNI, NOMBRE_APELLIDOS].
        birth_date_col (str): Birth date column (not used for linking).

    Returns:
        pd.DataFrame: The DataFrame with the new numeric 'PATIENTID' column.
    """
    
    df_result = df.copy()
    
    # Initialize temp_id with the original index. This serves as the unique ID for unlinked records.
    df_result['temp_id'] = np.arange(len(df_result))

    # --------------------------------------------------------------------------
    # 1. Apply Hierarchical Linking Logic (P1 and P2 ONLY)
    # --------------------------------------------------------------------------
    
    # --- PRIORITY 1: IDTARJETASANITARIA ---
    # Only link records that have a NON-NULL value in this column
    p1_col = id_columns[0]
    mask_p1_valid = pd.notna(df_result[p1_col])
    
    if mask_p1_valid.any():
        # Group by P1 and get minimum temp_id for each group
        p1_groups = df_result.loc[mask_p1_valid].groupby(p1_col)['temp_id'].transform('min')
        df_result.loc[mask_p1_valid, 'temp_id'] = p1_groups

    # --- PRIORITY 2: DNI ---
    # Only apply to records NOT linked by P1 and having NON-NULL DNI
    p2_col = id_columns[1]
    unlinked_mask = (df_result['temp_id'] == np.arange(len(df_result)))
    mask_p2_valid = unlinked_mask & pd.notna(df_result[p2_col])
    
    if mask_p2_valid.any():
        # Group by P2 and get minimum temp_id for each group
        p2_groups = df_result.loc[mask_p2_valid].groupby(p2_col)['temp_id'].transform('min')
        df_result.loc[mask_p2_valid, 'temp_id'] = p2_groups

    # --- PRIORITY 3: NOMBRE_APELLIDOS (NOT USED FOR LINKING) ---
    # Records still having temp_id == original index will remain as unique patients
    # This includes:
    # - Records with all IDs as None
    # - Records with only NOMBRE_APELLIDOS filled
    
    
    # --------------------------------------------------------------------------
    # 2. Unique Random Numeric ID Assignment
    # --------------------------------------------------------------------------
    
    # Get all unique patient groups
    unique_temp_ids = df_result['temp_id'].unique()
    
    # Generate unique random IDs (8 digits)
    min_range = 10_000_000
    max_range = 99_999_999
    
    # Safety check
    if len(unique_temp_ids) > (max_range - min_range + 1):
        raise ValueError("The number of unique patients exceeds the available random ID range.")
        
    # Generate unique random IDs
    random_ids = np.random.choice(
        range(min_range, max_range + 1), 
        size=len(unique_temp_ids), 
        replace=False
    )
    
    # Create mapping: temp_id -> random PATIENTID
    id_mapping = pd.Series(random_ids, index=unique_temp_ids)
    
    # Assign PATIENTID
    df_result['PATIENTID'] = df_result['temp_id'].map(id_mapping).astype('int64')

    # --------------------------------------------------------------------------
    # 3. Summary and Cleanup
    # --------------------------------------------------------------------------
    
    # Statistics
    final_count = df_result['PATIENTID'].nunique()
    total_records = len(df_result)
    
    # Count linking statistics
    p1_linked = (mask_p1_valid).sum() if 'mask_p1_valid' in locals() else 0
    p2_linked = (mask_p2_valid).sum() if 'mask_p2_valid' in locals() else 0
    unlinked = total_records - p1_linked - p2_linked
    
    print(f"\n{'='*60}")
    print(f"PATIENT ID ASSIGNMENT SUMMARY")
    print(f"{'='*60}")
    print(f"Total records processed: {total_records:,}")
    print(f"Total unique patients (PATIENTID): {final_count:,}")
    print(f"Average records per patient: {total_records / final_count:.2f}")
    print(f"\nLinking breakdown:")
    print(f"  - Linked by P1 (IDTARJETASANITARIA): {p1_linked:,} ({p1_linked/total_records*100:.1f}%)")
    print(f"  - Linked by P2 (DNI): {p2_linked:,} ({p2_linked/total_records*100:.1f}%)")
    print(f"  - Unlinked (unique IDs): {unlinked:,} ({unlinked/total_records*100:.1f}%)")
    print(f"{'='*60}\n")

    # Cleanup
    df_result = df_result.drop(columns=['temp_id'])
    
    return df_result

In [ ]:
# Call the hierarchical linking function
# The function will return a new DataFrame with the 'PATIENTID' column added.
df_edatoscaso_grouped_patientid = create_hierarchical_patient_id(
    df=df_edatoscaso_grouped,
    key_column=key_col,
    id_columns=id_columns
)

df_edatoscaso_grouped_patientid = dp.reorder_dataframe(df_edatoscaso_grouped_patientid)
dp.summarize_numbers(df_edatoscaso_grouped_patientid, key_col, id_columns)

# Convert data types

In [ ]:

def convert_objects_to_string(df):
    """
    Converts all columns in a DataFrame with 'object' dtype to the 
    dedicated pandas 'string' (StringDtype) type.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: The DataFrame with object columns converted.
    """
    # 1. Identify all columns with 'object' dtype
    object_cols = df.select_dtypes(include=['object']).columns

    if object_cols.empty:
        print("No 'object' columns found to convert.")
        return df

    print(f"Converting {len(object_cols)} 'object' columns to 'string'...")

    # 2. Convert the identified columns to the dedicated 'string' dtype
    # Use .loc to ensure we modify the original DataFrame correctly
    for col in object_cols:
        # We must use .loc to ensure the operation modifies the DataFrame correctly
        df.loc[:, col] = df[col].astype('string')
        
    print("Conversion complete.")
    return df

df_edatoscaso_grouped_patientid = convert_objects_to_string(df_edatoscaso_grouped_patientid)

In [ ]:
df_edatoscaso_grouped_patientid.info()

# Data Export

In [ ]:

df_edatoscaso_grouped_patientid.to_parquet(os.path.join(dp.source_tables_path, "tabla_madre/master_table.parquet"), index=False)

In [ ]:
df_edatoscaso_grouped_patientid.info()

# Quality Check

In [ ]:
# Loading exported table
master_table = pq.read_table(os.path.join(dp.source_tables_path, "tabla_madre/master_table.parquet")).to_pandas()
master_table.shape

In [ ]:
master_table.head(5)

In [ ]:
master_table.info()